In [2]:
# This notebook is 
#get common train/validation/test data
import pandas as pd
train_df = pd.read_csv("../covid/splits/train.csv")
val_df = pd.read_csv("../covid/splits/validation.csv")
test_df = pd.read_csv("../covid/splits/test.csv")

In [3]:
train_gen_df = train_df.copy()
val_gen_df = val_df.copy()
test_gen_df = test_df.copy()

train_gen_df["target"] = train_gen_df["target"].astype(str)
val_gen_df["target"] = val_gen_df["target"].astype(str)
test_gen_df["target"] = test_gen_df["target"].astype(str)

In [5]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.vgg16 import preprocess_input

BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)
COLOR_MODE = "rgb"
# Preprocessing adapté à VGG16
tl_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_generator_e4 = tl_datagen.flow_from_dataframe(
    dataframe=train_gen_df,
    x_col="filepath",
    y_col="target",
    target_size=IMAGE_SIZE,
    color_mode=COLOR_MODE,
    class_mode="binary",
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42
)

val_generator_e4 = tl_datagen.flow_from_dataframe(
    dataframe=val_gen_df,
    x_col="filepath",
    y_col="target",
    target_size=IMAGE_SIZE,
    color_mode=COLOR_MODE,
    class_mode="binary",
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_generator_e4 = tl_datagen.flow_from_dataframe(
    dataframe=test_gen_df,
    x_col="filepath",
    y_col="target",
    target_size=IMAGE_SIZE,
    color_mode=COLOR_MODE,
    class_mode="binary",
    batch_size=BATCH_SIZE,
    shuffle=False
)

Found 13545 validated image filenames belonging to 2 classes.
Found 3387 validated image filenames belonging to 2 classes.
Found 4233 validated image filenames belonging to 2 classes.


In [6]:
images_e4, labels_e4 = next(train_generator_e4)

print("Shape images :", images_e4.shape)
print("Shape labels :", labels_e4.shape)
print("Pixel min :", images_e4.min())
print("Pixel max :", images_e4.max())
print("Labels :", set(labels_e4))

Shape images : (32, 224, 224, 3)
Shape labels : (32,)
Pixel min : -123.68
Pixel max : 151.061
Labels : {np.float32(0.0), np.float32(1.0)}


In [7]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout

# Charge VGG16 pré-entraîné sur ImageNet.
# include_top=False retire son classifieur d'origine :
# on garde uniquement la partie qui extrait les caractéristiques des images.
base_model_e4 = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Feature extraction :
# on conserve les poids appris par VGG16 sans les modifier pendant l'entraînement.
base_model_e4.trainable = False

# On ajoute notre propre classifieur pour COVID / Non-COVID.
model_e4 = Sequential([
    base_model_e4,

    # Résume chaque feature map par sa moyenne.
    # Évite le très grand nombre de paramètres qu'aurait un Flatten.
    GlobalAveragePooling2D(),

    # Partie classification spécifique à notre problème.
    Dense(128, activation="relu"),
    Dropout(0.5),

    # Classification binaire : probabilité d'appartenir à la classe COVID.
    Dense(1, activation="sigmoid")
])

model_e4.summary()

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step 


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)                   │ (None, 7, 7, 512)           │      14,714,688 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 512)                 │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │          65,664 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │             129 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 14,780,481 (56.38 MB)

 Trainable params: 65,793 (257.00 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [8]:
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# Même loss, métriques et learning rate qu'E0
# pour faciliter la comparaison des expériences.
model_e4.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc")
    ]
)

# Arrête l'entraînement si val_loss ne s'améliore plus pendant 2 epochs.
# Puis restaure les poids correspondant à la meilleure val_loss.
early_stopping_e4 = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)



In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint_e4 = ModelCheckpoint(
    "E4_VGG16_best.keras",
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

In [9]:
# E4 = Transfer Learning seul :
# pas d'oversampling, pas de data augmentation.
# Le test n'est PAS utilisé pendant l'entraînement.
history_e4 = model_e4.fit(
    train_generator_e4,
    validation_data=val_generator_e4,
    epochs=5,
    callbacks=[early_stopping_e4,checkpoint_e4],
    verbose=1
)

Epoch 1/5
424/424 ━━━━━━━━━━━━━━━━━━━━ 2856s 7s/step - accuracy: 0.8936 - auc: 0.8956 - loss: 0.2846 - precision: 0.7501 - recall: 0.5657 - val_accuracy: 0.9203 - val_auc: 0.9740 - val_loss: 0.1897 - val_precision: 0.9640 - val_recall: 0.5544
Epoch 2/5
424/424 ━━━━━━━━━━━━━━━━━━━━ 405221s 958s/step - accuracy: 0.9346 - auc: 0.9642 - loss: 0.1669 - precision: 0.8624 - recall: 0.7342 - val_accuracy: 0.9504 - val_auc: 0.9820 - val_loss: 0.1252 - val_precision: 0.9438 - val_recall: 0.7547
Epoch 3/5
424/424 ━━━━━━━━━━━━━━━━━━━━ 3285s 8s/step - accuracy: 0.9453 - auc: 0.9713 - loss: 0.1466 - precision: 0.8809 - recall: 0.7861 - val_accuracy: 0.9531 - val_auc: 0.9870 - val_loss: 0.1192 - val_precision: 0.9688 - val_recall: 0.7496
Epoch 4/5
424/424 ━━━━━━━━━━━━━━━━━━━━ 3389s 8s/step - accuracy: 0.9540 - auc: 0.9804 - loss: 0.1228 - precision: 0.8990 - recall: 0.8232 - val_accuracy: 0.9678 - val_auc: 0.9890 - val_loss: 0.0942 - val_precision: 0.9167 - val_recall: 0.8929
Epoch 5/5
424/424 ━━━━━━

In [1]:
# Sauvegarde E4 avec les meilleurs poids restaurés par EarlyStopping
model_e4.save("E4_VGG16_TL_baseline.keras") 

NameError: name 'model_e4' is not defined